# SpeakAI-Eval — Inference (Run Only)

Notebook này chỉ cần gọi lên và chạy. Đảm bảo bạn đã chạy **Setup Notebook** trước đó.
Mã nguồn, models và HF cache sẽ được đọc thẳng từ Google Drive.

In [ ]:
# Cài đặt thư viện
import subprocess, sys, os
pkgs = [
    'torch>=2.1.0', 'torchaudio>=2.1.0', 'torch-geometric>=2.4.0',
    'transformers>=4.36.0', 'peft>=0.7.0', 'pyyaml>=6.0.1',
    'numpy>=1.24.0', 'soundfile>=0.12.1', 'nltk>=3.8.1',
    'noisereduce>=3.0.0', 'python-dotenv>=1.0.0',
    'speechbrain>=1.0.0', 'huggingface_hub>=0.23.0', 'silero-vad>=6.2.1',
    'accelerate>=0.26.0', 'fastapi', 'uvicorn', 'python-multipart',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
print('Libraries installed!')
if not os.path.exists('/usr/local/bin/cloudflared'):
    print('Downloading cloudflared...')
    subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared', shell=True, check=True)
    subprocess.run('chmod +x /usr/local/bin/cloudflared', shell=True, check=True)


---
## Khởi tạo Pipeline từ Drive

In [ ]:
import sys, os
SOURCE_DIR = None
MODEL_DIR = None
PRONUNCIATION_PATH = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'pretrained_models' in dirs and 'wavlm-large' in os.listdir(os.path.join(root, 'pretrained_models')):
        MODEL_DIR = root
    if 'infer' in dirs and 'ecapa-diarize' in dirs:
        SOURCE_DIR = root
    if 'pronunciation.pt' in files:
        PRONUNCIATION_PATH = os.path.join(root, 'pronunciation.pt')
if not MODEL_DIR:
    raise FileNotFoundError("Không tìm thấy thư mục 'pretrained_models/wavlm-large' trong dataset model.")
if not SOURCE_DIR:
    raise FileNotFoundError("Không tìm thấy source code (infer, ecapa-diarize) trong dataset setup.")
if not PRONUNCIATION_PATH:
    raise FileNotFoundError("Không tìm thấy file 'pronunciation.pt' (mô hình speechocean) trong bất kỳ dataset nào.")
sys.path.insert(0, SOURCE_DIR)
sys.path.insert(0, f'{SOURCE_DIR}/ecapa-diarize')
os.chdir(SOURCE_DIR)

# Lưu cache HuggingFace vào MODEL_DIR (nếu có thể) hoặc /kaggle/working
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'

# VÁ LỖI GOOGLE DRIVE READ-ONLY (monkey-patch)
import ecapa_diarize.embedding
import tempfile
original_init = ecapa_diarize.embedding.EcapaEmbedder.__init__
def patched_init(self, device="cpu"):
    from speechbrain.inference.speaker import SpeakerRecognition
    from speechbrain.utils.fetching import LocalStrategy
    from ecapa_diarize.paths import ECAPA_DIR
    cache_dir = os.path.join(tempfile.gettempdir(), "ecapa_cache")
    os.makedirs(cache_dir, exist_ok=True)
    self.device = device
    import re
    hp_path = f'{ECAPA_DIR}/hyperparams.yaml'
    with open(hp_path, 'r', encoding='utf-8') as f:
        content = f.read()
    content = re.sub(r'pretrained_path:\s*.*', f'pretrained_path: {ECAPA_DIR}', content)
    tmp_hp = f'{cache_dir}/hyperparams.yaml'
    with open(tmp_hp, 'w', encoding='utf-8') as f:
        f.write(content)
    self._model = SpeakerRecognition.from_hparams(
        source=str(ECAPA_DIR),
        hparams_file=tmp_hp,
        savedir=cache_dir,
        run_opts={"device": device},
        local_strategy=LocalStrategy.COPY,
    )
ecapa_diarize.embedding.EcapaEmbedder.__init__ = patched_init

import torch, yaml
from infer.pipeline import SpeakingPipeline
from transformers import AutoModelForCausalLM, AutoTokenizer

print('Patching config dynamically...')
with open(f'{SOURCE_DIR}/configs/pronunciation.yaml', 'r') as f:
    cfg = yaml.safe_load(f)
cfg['asr']['model_name'] = f'{MODEL_DIR}/pretrained_models/whisper-large-v3-turbo'
cfg['wavlm']['model_name'] = f'{MODEL_DIR}/pretrained_models/wavlm-large'
cfg['paths']['pronunciation_checkpoint'] = PRONUNCIATION_PATH
os.makedirs('/kaggle/working/tmp_configs', exist_ok=True)
tmp_config = '/kaggle/working/tmp_configs/pronunciation.yaml'
with open(tmp_config, 'w') as f:
    yaml.dump(cfg, f)

print('Patching transcribe.py dynamically...')
import infer.transcribe
def patched_load_asr_config():
    with open(tmp_config, encoding='utf-8') as f:
        return yaml.safe_load(f).get('asr') or {}
infer.transcribe._load_asr_config = patched_load_asr_config

print('Loading Pipeline models on GPU...')
pipeline = SpeakingPipeline(config_path=tmp_config, device='cuda')
print('Pipeline ready!')

model_name = f'{MODEL_DIR}/pretrained_models/Qwen2.5-3B-Instruct'
print(f'Loading LLM from Kaggle: {model_name}')
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map='cuda'
)
print(f'LLM Ready on {llm_model.device}!')


---
## Hàm Sinh Feedback Tổng Hợp bằng Qwen LLM

In [ ]:
def generate_overall_feedback(result):
    student_sents = result.get('student', {}).get('sentences', [])
    if not student_sents:
        return 'Không có dữ liệu học viên để đánh giá.'
    
    total_acc = sum(s.get('scores', {}).get('accuracy', 0) for s in student_sents) / len(student_sents)
    total_flu = sum(s.get('scores', {}).get('fluency', 0) for s in student_sents) / len(student_sents)
    total_pro = sum(s.get('scores', {}).get('prosodic', 0) for s in student_sents) / len(student_sents)
    overall_total = (total_acc + total_flu + total_pro) / 3
    
    dialogue_turns = result.get('dialogue', {}).get('turns', [])
    conversation_text = ''
    bad_words_all = []
    bad_ph_all = []
    
    for turn in dialogue_turns:
        speaker = turn.get('role', '').upper()
        transcript = turn.get('transcript', '')
        if speaker == 'TEACHER':
            conversation_text += f'Giáo viên: {transcript}\n'
        elif speaker == 'STUDENT':
            turn_score = turn.get('scores', {}).get('accuracy', 0)
            conversation_text += f'Học viên: {transcript} (Điểm phát âm: {turn_score:.1f}/10)\n'
            errors = turn.get('errors', {})
            bad_words = [w for w in errors.get('words', []) if w.get('score', 10) < 7.0]
            bad_ph = [p for p in errors.get('phonemes', []) if p.get('score', 10) < 7.0]
            bad_words_all.extend([f"{w['word']} ({w.get('score',0):.1f})" for w in bad_words])
            bad_ph_all.extend([f"{p['phoneme']} ({p.get('score',0):.1f})" for p in bad_ph])
    
    bad_words_str = ', '.join(list(dict.fromkeys(bad_words_all))[:15]) or 'Không có'
    bad_ph_str = ', '.join(list(dict.fromkeys(bad_ph_all))[:15]) or 'Không có'
    
    prompt = f"""Bạn là một giáo viên chuyên đánh giá phát âm và giao tiếp tiếng Anh.\nDưới đây là đoạn hội thoại giữa Giáo viên và Học viên, cùng với kết quả phân tích phát âm của Học viên. Hãy viết một bài nhận xét TỔNG HỢP (Overall Feedback) thật chi tiết, rõ ràng và có cấu trúc dễ đọc.\n\n--- ĐOẠN HỘI THOẠI ---\n{conversation_text}\n--- ĐIỂM SỐ TRUNG BÌNH CỦA HỌC VIÊN (Thang 10) ---\nTổng quan: {overall_total:.1f}\nChính xác (Accuracy): {total_acc:.1f}\nTrôi chảy (Fluency): {total_flu:.1f}\nNgữ điệu (Prosody): {total_pro:.1f}\n\n--- LỖI PHÁT ÂM ĐÁNG CHÚ Ý CẦN SỬA ---\nTừ phát âm sai nhiều: {bad_words_str}\nÂm vị (Phoneme) sai nhiều: {bad_ph_str}\n\nYêu cầu nhận xét (bằng tiếng Việt, định dạng Markdown đẹp, rõ ràng):\n1. Đánh giá chung: Học viên làm tốt ở đâu (khen ngợi), giao tiếp có tự nhiên và đúng ngữ cảnh không? Ngữ pháp sử dụng có đúng không?\n2. Điểm cần khắc phục: Giải thích thật rõ ràng các lỗi phát âm (từ/âm vị cụ thể) và hướng dẫn cách sửa chi tiết.\n3. Lời khuyên & Động viên: Đề xuất cách luyện tập để cải thiện.\nKhông nhắc đến 'Completeness'.\n"""
    messages = [
        {'role': 'system', 'content': 'Bạn là giáo viên tiếng Anh tận tâm, chuyên môn cao.'},
        {'role': 'user', 'content': prompt}
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response


---
## Khởi chạy Backend API (FastAPI + Cloudflare Tunnel)

In [ ]:
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from fastapi.responses import JSONResponse
import uvicorn
import shutil
import json
import numpy as np
import subprocess
import time
import re
import os
import uuid
import threading

# Khởi tạo Global Lock để đảm bảo GPU chỉ chạy 1 Request tại một thời điểm (tránh OOM Crash)
gpu_lock = threading.Lock()

# 1. Khởi động Cloudflare Tunnel
def start_cloudflare_tunnel(port=8000):
    print('Starting Cloudflare Tunnel...')
    cmd = f'cloudflared tunnel --url http://127.0.0.1:{port}'
    process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    url = None
    for _ in range(20):
        line = process.stdout.readline()
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            url = match.group(0)
            break
        time.sleep(0.5)
    return url

PUBLIC_URL = start_cloudflare_tunnel(8000)
print('\n' + '='*80)
print(f'🚀 API IS LIVE AT: {PUBLIC_URL}')
print('=> COPY LINK NÀY VÀ DÁN VÀO CẤU HÌNH TRÊN WEBSITE CỦA BẠN!')
print('='*80 + '\n')

# 2. Khởi tạo FastAPI App
app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

# Phục vụ file audio tĩnh từ Colab để Website có thể nghe lại
os.makedirs('/tmp/SpeakAI_Audio', exist_ok=True)
app.mount('/audio', StaticFiles(directory='/tmp/SpeakAI_Audio'), name='audio')

@app.post('/assess')
def assess_api(audio: UploadFile = File(...), teacher_embeddings_json: str = Form(...), student_embeddings_json: str = Form(...), score_teacher: bool = Form(False)):
    try:
        req_id = uuid.uuid4().hex
        # Lưu audio hội thoại theo UUID để tránh bị đè khi có nhiều người dùng
        conv_path = f'/tmp/SpeakAI_Audio/{req_id}_{audio.filename}'
        with open(conv_path, 'wb') as f:
            shutil.copyfileobj(audio.file, f)
        
        # Parse Embeddings của Giáo viên
        t_emb_list = json.loads(teacher_embeddings_json)
        teacher_emb = np.array(t_emb_list, dtype=np.float32)
        if len(teacher_emb.shape) == 2:
            teacher_emb = np.mean(teacher_emb, axis=0)
        teacher_emb /= np.linalg.norm(teacher_emb)

        # Parse Embeddings của Học viên
        s_emb_list = json.loads(student_embeddings_json)
        student_emb = np.array(s_emb_list, dtype=np.float32)
        if len(student_emb.shape) == 2:
            student_emb = np.mean(student_emb, axis=0)
        student_emb /= np.linalg.norm(student_emb)
        
        # Gọi Pipeline với GPU Lock để chống OOM khi chạy song song
        print(f'[{req_id}] Waiting for GPU lock...')
        with gpu_lock:
            print(f'[{req_id}] Running Pipeline...')
            raw_result = pipeline.assess_conversation(
                conv_path,
                teacher_embedding=teacher_emb,
                student_embedding=student_emb,
                score_teacher=score_teacher
            )
            
            print(f'[{req_id}] Generating LLM Feedback...')
            llm_feedback = generate_overall_feedback(raw_result)
        
        # Trích xuất file tổng hợp
        diar = raw_result.get('diarization', {})

        if raw_result.get('teacher') and diar.get('teacher'):
            raw_result['teacher']['full_audio'] = str(diar['teacher'])
        if raw_result.get('student') and diar.get('student'):
            raw_result['student']['full_audio'] = str(diar['student'])

        # Chuyển đổi đường dẫn file cục bộ thành Public URL
        def convert_paths_to_urls(node):
            if isinstance(node, dict):
                for k, v in node.items():
                    if (k == 'audio' or k == 'full_audio') and isinstance(v, str) and v.startswith('/tmp/SpeakAI_Audio/'):
                        rel_path = v.replace('/tmp/SpeakAI_Audio/', '')
                        node[k] = f'{PUBLIC_URL}/audio/{rel_path}'
                    else:
                        convert_paths_to_urls(v)
            elif isinstance(node, list):
                for item in node:
                    convert_paths_to_urls(item)
                    
        convert_paths_to_urls(raw_result)
        
        # Xóa file audio tạm
        if os.path.exists(conv_path):
            os.remove(conv_path)
            
        return JSONResponse({
            'success': True,
            'result': raw_result,
            'llm_feedback': llm_feedback
        })
    except Exception as e:
        print(f'API Error: {e}')
        import traceback
        traceback.print_exc()
        return JSONResponse({'success': False, 'error': str(e)}, status_code=500)

# Khởi chạy Uvicorn
config = uvicorn.Config(app, host='0.0.0.0', port=8000)
server = uvicorn.Server(config)
await server.serve()
